# DS 227 &middot; Knowledge Discovery in Data &mdash; Week 11 Lab
## Auditing a Dataset for Privacy Risk

Data about people carries risk. This week you learn to spot **identifiers** and
**quasi-identifiers**, test for re-identification, and reduce the risk before sharing.

**How long:** about 45 minutes. Nothing to install.

Work top to bottom. The Stretch section at the end is optional.

---
## Part 0 &middot; Name the identifiers

A **direct identifier** names a person (id, email). A **quasi-identifier** (age, zip,
sex) does not alone, but combined can. Run the cell.

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "student_id": ["2024-001","2024-002","2024-003","2024-004"],
    "age": [19, 19, 21, 19],
    "barangay": ["Lahug","Lahug","Apas","Lahug"],
    "gpa": [1.75, 2.00, 1.25, 1.50],
})
print(df)

**Answer here** (double-click to edit):

1. Which column is a *direct* identifier, and which columns are *quasi*-identifiers that
   could combine to single someone out?
   &rarr; *your answer*

2. `gpa` is sensitive but not identifying on its own. Why does the risk come from the quasi-
   identifiers *around* it rather than the sensitive value itself?
   &rarr; *your answer*

---
## Part 1 &middot; Test for re-identification (k-anonymity)

Group by the quasi-identifiers. Any group of size **1** is a person who can be singled
out. Run the cell.

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "age": [19, 19, 21, 19],
    "barangay": ["Lahug","Lahug","Apas","Lahug"],
    "gpa": [1.75, 2.00, 1.25, 1.50],
})
sizes = df.groupby(["age", "barangay"]).size()
print(sizes)
print("\nunique (k=1) groups:\n", sizes[sizes == 1])

**Answer here:**

1. Which `(age, barangay)` combination has only one person? Why does a group of size 1 make
   that person re-identifiable even without their id?
   &rarr; *your answer*

2. "k-anonymity" means every combination appears at least *k* times. What is the smallest
   group size here, and is the dataset 2-anonymous?
   &rarr; *your answer*

---
## Part 2 &middot; Reduce the risk: drop and generalise

Two moves: **remove** direct identifiers, and **generalise** quasi-identifiers (e.g. age
into bands) so groups grow. Run the cell.

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "student_id": ["2024-001","2024-002","2024-003","2024-004"],
    "age": [19, 19, 21, 19],
    "barangay": ["Lahug","Lahug","Apas","Lahug"],
    "gpa": [1.75, 2.00, 1.25, 1.50],
})
safe = df.drop(columns=["student_id"]).copy()
safe["age_band"] = pd.cut(safe["age"], bins=[17, 20, 25], labels=["18-20", "21-25"])
safe = safe.drop(columns=["age"])
print(safe.groupby(["age_band", "barangay"], observed=True).size())

**Answer here:**

1. After dropping the id and banding age, did the smallest group get bigger? What did
   generalisation buy you?
   &rarr; *your answer*

2. Generalising loses detail. What is the trade-off between privacy and analytical
   usefulness, and who should decide where to draw the line?
   &rarr; *your answer*

---
## Part 3 &middot; Aggregates leak too

Even summaries can expose a person if a group is tiny. Never report a statistic for a
group of one. Run the cell.

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "barangay": ["Lahug","Lahug","Lahug","Apas"],
    "gpa": [1.75, 2.00, 1.50, 1.25],
})
counts = df.groupby("barangay")["gpa"].agg(["mean", "count"])
print(counts)
print("\nsafe to publish (count >= 3):\n", counts[counts["count"] >= 3])

**Answer here:**

1. The mean GPA for Apas *is* that one person's GPA. Why does publishing a group mean of
   size 1 leak private information?
   &rarr; *your answer*

2. A common rule is to suppress any cell with fewer than N people. Why do statistical
   agencies enforce a minimum group size before releasing a figure?
   &rarr; *your answer*

---
## Stretch &mdash; optional

Stop here if you like; the required part is done.

### Stretch 1 &middot; Hash an identifier

Sometimes you must keep a stable id without revealing it. Replace `student_id` with a
`hashlib.sha256` hex digest. Does hashing make it *anonymous*, or just *pseudonymous*?
Explain the difference.

In [ ]:
# your code here

### Stretch 2 &middot; Find the riskiest column combo

Try grouping by different pairs of quasi-identifiers. Which combination produces the
most size-1 groups &mdash; i.e. carries the highest re-identification risk?

In [ ]:
# your code here

---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to
download.

You need a **submit token** &mdash; one covers every lab for a month. Open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token), sign in and generate it,
then add it **once** to Colab's Secrets panel (the &#128273; icon, left sidebar) as
`LATARAK_TOKEN`. After that the cell reads it automatically, with no prompt. No Secrets
panel? The cell will just ask, hiding what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "ds227", 11

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/ds227/lab/11/submit"
    )

# The LIVE notebook, including edits you have not saved yet.
nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]

# A month-long token. Store it once in Colab Secrets (key LATARAK_TOKEN) and
# this reads it with no prompt; otherwise it asks and hides what you type.
try:
    from google.colab import userdata
    token = (userdata.get("LATARAK_TOKEN") or "").strip()
except Exception:
    token = ""
if not token:
    token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 11 submission page](https://portal.latarak.com/course/ds227/lab/11/submit) and upload it.

Re-submitting replaces your previous attempt; the most recent version is the one kept.